# 04 · 1D CNN training

Same window split as the RF baseline so the comparison is fair. Inputs go in as `(batch, 10, 20)` — channels first.

If you're running this on Colab, uncomment the install + drive-mount cell below.

In [ ]:
# Colab only — uncomment if running here.
# !pip install -q torch scikit-learn matplotlib seaborn tqdm
# from google.colab import drive
# drive.mount('/content/drive')
# ROOT = '/content/drive/MyDrive/emg-gesture-classification'

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path(ROOT) if 'ROOT' in globals() and isinstance(ROOT, str) else (Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd())
sys.path.insert(0, str(ROOT))

import numpy as np
import torch

from src.models import EMG1DCNN, count_parameters
from src.train import TrainConfig, pick_device, predict, train_model
from src.preprocess import normalize_per_channel
from src.evaluate import per_class_report, plot_confusion_matrix, save_metrics

device = pick_device()
print('device:', device)

In [ ]:
DATA = np.load(ROOT / 'data' / 'processed' / 'windows.npz')
windows, labels, reps = DATA['windows'], DATA['labels'], DATA['reps']

train_mask = np.isin(reps, [1, 2, 3, 4, 5, 6])
val_mask   = np.isin(reps, [7, 8])
test_mask  = np.isin(reps, [9, 10])

X_train_raw = windows[train_mask]
X_train, stats = normalize_per_channel(X_train_raw)
X_val,   _ = normalize_per_channel(windows[val_mask],  stats=stats)
X_test,  _ = normalize_per_channel(windows[test_mask], stats=stats)
y_train, y_val, y_test = labels[train_mask], labels[val_mask], labels[test_mask]

n_classes = int(labels.max()) + 1
print('n_classes:', n_classes, 'train:', X_train.shape)

In [ ]:
model = EMG1DCNN(n_channels=X_train.shape[-1], n_classes=n_classes)
print(f'parameters: {count_parameters(model):,}')

cfg = TrainConfig(
    epochs=50, batch_size=64, lr=1e-3, weight_decay=1e-4, patience=8,
    checkpoint_path=str(ROOT / 'results' / 'metrics' / 'cnn_best.pt'),
)
model, history = train_model(model, (X_train, y_train), (X_val, y_val), cfg=cfg, device=device)

np.savez(ROOT / 'results' / 'metrics' / 'norm_stats.npz', **stats)

In [ ]:
test_probs = predict(model, X_test, device=device)
test_pred = test_probs.argmax(axis=1)

cnn_metrics = per_class_report(y_test, test_pred)
save_metrics(cnn_metrics, ROOT / 'results' / 'metrics' / 'cnn_metrics.json')
print('test accuracy:', cnn_metrics['overall']['accuracy'])
plot_confusion_matrix(y_test, test_pred,
                      title='1D CNN — test confusion matrix',
                      save_path=ROOT / 'results' / 'figures' / 'cnn_confusion.png')